In [1]:
from pathlib import Path

import torch
import torch.nn.functional as F
from torch import nn

from torch.utils.data import DataLoader, random_split

from tts.jenny_dataset import JennyDataset
from tts.model import PhonemeToSpectrogram, build_vocab
from tts.tokenizer import phonemize  # your function: str -> list[str]

In [2]:
def get_device():
    if torch.cuda.is_available():   return torch.device("cuda")
    if torch.backends.mps.is_available(): return torch.device("mps")
    return torch.device("cpu")

In [3]:
# def collate_fn(batch):
#     """
#     batch: list of (mel, text)
#       mel:  (1, n_mels, T)  float32
#       text: str
#     Returns:
#       mels:  (B, n_mels, T_max)  zero-padded along time
#       texts: list[str]
#     """
#     mels, texts = zip(*batch)
#     mels  = [m.squeeze(0) for m in mels]          # → (n_mels, T) each
#     max_t = max(m.shape[-1] for m in mels)
#     padded = torch.stack([F.pad(m, (0, max_t - m.shape[-1])) for m in mels])
#     return padded, list(texts)

In [4]:
# ── Device ────────────────────────────────────────────────────────────────────

device = get_device()
print(f"device: {device}")

# ── Model ─────────────────────────────────────────────────────────────────────

vocab = build_vocab()

model = PhonemeToSpectrogram(
    vocab_size   = len(vocab),
    d_model      = 256,
    nhead        = 4,
    num_layers   = 4,
    n_mels       = 100,   # Vocos uses 100
    patch_frames = 48,
).to(device)

# Initialise with better weights for the time bit (equally spaced time bins)
nn.init.zeros_(model.dur_head.net[-1].weight)
nn.init.constant_(model.dur_head.net[-1].bias, 16.0) # seperate tokens by 16*256/24000 to start with ~ 0.2s


optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

print(f"params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")



device: mps
params: 4,518,081


In [5]:
# ── Data ──────────────────────────────────────────────────────────────────────
data_fpath = '/Users/dominicbates/Documents/Github/SimpleTTS/data/jenny'
full_ds = JennyDataset(data_fpath)
val_n   = int(len(full_ds) * 0.05)
trn_ds, val_ds = random_split(
    full_ds, [len(full_ds) - val_n, val_n],
    generator=torch.Generator().manual_seed(42),
)

def collate_fn(batch):
    mels, texts = zip(*batch)
    mels  = [m.squeeze(0) for m in mels]           # (n_mels, T) each
    max_t = max(m.shape[-1] for m in mels)
    padded = torch.stack([F.pad(m, (0, max_t - m.shape[-1])) for m in mels])
    return padded, list(texts)                      # (B, n_mels, T_max), list[str]

trn_loader = DataLoader(trn_ds, batch_size=16, shuffle=True,  collate_fn=collate_fn, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=16, shuffle=False, collate_fn=collate_fn, num_workers=0)

print(f"train={len(trn_ds)}  val={len(val_ds)}")


[JennyDataset] 20978 samples found in /Users/dominicbates/Documents/Github/SimpleTTS/data/jenny/samples
train=19930  val=1048


In [6]:
# ── Text → token ids ──────────────────────────────────────────────────────────

def texts_to_ids(texts):
    all_ids = []
    for text in texts:
        tokens = phonemize(text)
        all_ids.append([vocab.get(t, vocab["<UNK>"]) for t in tokens])

    max_len   = max(len(ids) for ids in all_ids)
    pad_idx   = vocab["<PAD>"]
    id_tensor = torch.full((len(all_ids), max_len), pad_idx, dtype=torch.long, device=device)
    for i, ids in enumerate(all_ids):
        id_tensor[i, :len(ids)] = torch.tensor(ids, dtype=torch.long)

    padding_mask = (id_tensor == pad_idx)           # True = ignore
    return id_tensor, padding_mask

# ── Train / val step ──────────────────────────────────────────────────────────

def train_step(mels, texts):
    model.train()
    mels = mels.to(device)
    token_ids, padding_mask = texts_to_ids(texts)
    loss = model.compute_loss(token_ids, mels, padding_mask=padding_mask)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    return loss.item()

@torch.no_grad()
def val_step(mels, texts):
    model.eval()
    mels = mels.to(device)
    token_ids, padding_mask = texts_to_ids(texts)
    return model.compute_loss(token_ids, mels, padding_mask=padding_mask).item()


In [7]:
EVAL_TEXTS = [
    "The quick brown fox jumps over the lazy dog",
    "Oh my god! That is amazing!",
    "LOL haha that's mental dude!",
    "Can I ask you a question?",
    "Let's try a really long string to see what happens. And maybe a second sentence too?"
    "I like pizza, dogs, beer, and making long lists",
    "The dog said \"woof woof!\"",
    "abcdefghijklmnop",
    "supercalifragilisticexpialidocious",
    # "",
]
 
RUN_DIR = Path("./runs/run_004")   # change for each experiment
 
 
def save_eval_specs(epoch):
    """Synthesise each eval text and save to RUN_DIR/specs/epoch_NNN_N.pt"""
    spec_dir = RUN_DIR / "specs"
    spec_dir.mkdir(parents=True, exist_ok=True)
    model.eval()
    with torch.no_grad():
        for i, text in enumerate(EVAL_TEXTS):
            token_ids, padding_mask = texts_to_ids([text])
            spec = model.synthesise(token_ids, padding_mask=padding_mask)  # (n_mels, T)
            torch.save(
                {"spec": spec.cpu(), "text": text, "epoch": epoch},
                spec_dir / f"epoch_{epoch:03d}_{i:02d}.pt",
            )
    print(f"  [eval] specs saved to {spec_dir}")
    model.train()
 
     
def load_eval_spec(epoch, sample_idx=0):
    """
    Load a saved eval spectrogram back as a float32 tensor.
 
    Returns:
        spec: (n_mels, T)  float32  — ready to pass straight to the vocoder
        text: str
    """
    path = RUN_DIR / "specs" / f"epoch_{epoch:03d}_{sample_idx:02d}.pt"
    data = torch.load(path, weights_only=True)
    return data["spec"].float(), data["text"]
 

In [ ]:
from tqdm import tqdm

EPOCHS     = 30
SAVE_EVERY = 2

(RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)

for epoch in range(1, EPOCHS + 1):

    trn_losses = []
    for m, t in tqdm(trn_loader, desc=f"epoch {epoch:03d} train", leave=False):
        trn_losses.append(train_step(m, t))

    val_losses = []
    for m, t in tqdm(val_loader, desc=f"epoch {epoch:03d} val  ", leave=False):
        val_losses.append(val_step(m, t))


    # trn_losses = []
    # for i, (m, t) in enumerate(tqdm(trn_loader, desc=f"epoch {epoch:03d} train", leave=False)):
    #     if i >= 70: break
    #     trn_losses.append(train_step(m, t))

    # val_losses = []
    # for i, (m, t) in enumerate(tqdm(val_loader, desc=f"epoch {epoch:03d} val  ", leave=False)):
    #     if i >= 5: break
    #     val_losses.append(val_step(m, t))

    trn_loss = sum(trn_losses) / len(trn_losses)
    val_loss = sum(val_losses) / len(val_losses)
    print(f"epoch {epoch:03d}  train={trn_loss:.4f}  val={val_loss:.4f}")

    save_eval_specs(epoch)
    if epoch % SAVE_EVERY == 0:
        torch.save(
            {"epoch": epoch, "model": model.state_dict(),
             "optimizer": optimizer.state_dict(), "val_loss": val_loss},
            RUN_DIR / "checkpoints" / f"epoch_{epoch:03d}.pt",
        )

epoch 001 train:   3%|█                                 | 39/1246 [01:07<38:58,  1.94s/it]

In [ ]:
save_eval_specs(epoch)

In [2]:
# for text in EVAL_TEXTS:
#     tokens = phonemize(text)
#     print(repr(text), "→", tokens)
# After a forward pass, inspect what positions look like
model.eval()
with torch.no_grad():
    token_ids, padding_mask = texts_to_ids(["Hello how are you today"])
    patches, positions = model(token_ids, padding_mask=padding_mask)
    print(positions)
    print("min:", positions.min().item(), "max:", positions.max().item())
    print("diffs:", (positions[0, 1:] - positions[0, :-1]))

NameError: name 'model' is not defined

In [18]:
len(['HH',
 'AH0',
 'L',
 'OW1',
 '|',
 'DH',
 'EH1',
 'R',
 '|',
 '<PERIOD>',
 'HH',
 'AW1',
 'Z',
 '|',
 'IH1',
 'T',
 '|',
 'G',
 'OW1',
 'IH0',
 'NG',
 '|',
 '<QUESTION>'])

23

In [20]:
I would expect this to be around: ~4 seconds
We have 23 of my tokens there (using my tokenizer to phonemes



SyntaxError: invalid syntax (3981927091.py, line 1)

In [21]:
import torch.nn.functional as F
F.softplus(torch.tensor(2.8))  # should print ~16

tensor(2.8590)

In [ ]:
This prints 2.8590